# Milestone 6: Multimodal Driver Distraction Explainable AI (SHAP)

**Project**: Multimodal Driver Distraction Detection System  
**Goal**: Interpret and validate the fine-tuned `ResidualMLP` multimodal classifier predictions using SHAP (SHapley Additive exPlanations).  
**Key Features**: Global & Local Feature Attribution, Class-Wise SHAP Analysis, Modality Contribution Share (Visual Embeddings vs. Geometric Landmarks).

In [ ]:
# 1. Environment Setup & Dependency Installation
try:
    from google.colab import drive
    IN_COLAB = True
    drive.mount("/content/drive")
except ImportError:
    IN_COLAB = False
    print("Running in local environment.")

!pip install -q shap matplotlib seaborn pandas numpy torch scikit-learn

In [ ]:
# 2. Imports & Path Auto-Detection
import os
import sys
import json
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.preprocessing import LabelEncoder
import shap

# Seed reproducibility
SEED = 42
rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)

# Path Auto-Detection
def find_path(rel_path):
    candidates = [
        Path(rel_path),
        Path("/content") / rel_path,
        Path("/content/drive/MyDrive") / rel_path,
        Path("/content/drive/MyDrive/RI") / rel_path,
        Path(os.getcwd()) / rel_path,
        Path(os.getcwd()).parent / rel_path
    ]
    for p in candidates:
        if p.exists():
            return p
    return Path(rel_path)

M4_DIR = find_path("Milestone4_Outputs")
M5_DIR = find_path("Milestone5_Outputs")
OUTPUT_DIR = Path("Milestone6_Outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"M4 Path: {M4_DIR}")
print(f"M5 Path: {M5_DIR}")
print(f"M6 Output Path: {OUTPUT_DIR}")

In [ ]:
# 3. Data Loading & Subject-Wise Verification
fused_features = np.load(M4_DIR / "fusion_features.npy")
fused_labels = np.load(M4_DIR / "fusion_labels.npy", allow_pickle=True)
fused_subjects = np.load(M4_DIR / "fusion_subjects.npy", allow_pickle=True)

fused_features = np.nan_to_num(fused_features, nan=0.0)
le = LabelEncoder()
y_encoded = le.fit_transform(fused_labels)
class_names = [str(cls) for cls in le.classes_]

class_descriptions = {
    "c0": "Safe Driving", "c1": "Texting - Right", "c2": "Talking - Right",
    "c3": "Texting - Left", "c4": "Talking - Left", "c5": "Operating Radio",
    "c6": "Drinking", "c7": "Reaching Behind", "c8": "Hair & Makeup", "c9": "Talking to Passenger"
}
full_class_names = [f"{c}: {class_descriptions.get(c, c)}" for c in class_names]

# Subject-wise split matching Milestone 5
unique_subs = sorted(list(set(fused_subjects)))
split_idx = int(len(unique_subs) * 0.8)
train_subs = unique_subs[:split_idx]
val_subs = unique_subs[split_idx:]
train_mask = np.isin(fused_subjects, train_subs)
val_mask = np.isin(fused_subjects, val_subs)

X_train, y_train = fused_features[train_mask], y_encoded[train_mask]
X_val, y_val = fused_features[val_mask], y_encoded[val_mask]

geo_feature_names = ["face_xmin", "face_ymin", "face_width", "face_height", "pitch", "yaw", "roll", "gaze_x", "gaze_y", "ear_avg", "eye_open_left", "eye_open_right"]
vis_feature_names = [f"vis_embed_{i}" for i in range(320)]
feature_names = vis_feature_names + geo_feature_names

print(f"Total Samples: {len(fused_features)}, Feature Dim: {fused_features.shape[1]}")
print(f"Train Set: {len(X_train)} samples ({len(train_subs)} subjects)")
print(f"Val Set: {len(X_val)} samples ({len(val_subs)} subjects)")

In [ ]:
# 4. ResidualMLP Architecture Definition & Checkpoint Loading
class ResidualBlock(nn.Module):
    def __init__(self, dim, dropout=0.3):
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(dim, dim), nn.BatchNorm1d(dim), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(dim, dim)
        )
        self.relu = nn.ReLU()
    def forward(self, x):
        return self.relu(self.block(x) + x)

class MultimodalClassifier(nn.Module):
    def __init__(self, input_dim, num_classes=10):
        super().__init__()
        self.input_layer = nn.Sequential(
            nn.Linear(input_dim, 512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(0.30)
        )
        self.res_block = ResidualBlock(512, dropout=0.30)
        self.transition = nn.Sequential(
            nn.Linear(512, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.30)
        )
        self.classifier = nn.Linear(256, num_classes)
    def forward(self, x):
        x = self.input_layer(x)
        x = self.res_block(x)
        x = self.transition(x)
        return self.classifier(x)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MultimodalClassifier(input_dim=fused_features.shape[1], num_classes=len(class_names)).to(device)
ckpt_path = M5_DIR / "best_classifier.pth"
# Note: weights_only=False handled for PyTorch 2.6+ checkpoint compatibility
ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()
print(f"Loaded model successfully from {ckpt_path}. Parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# 5. SHAP DeepExplainer Computation
bg_idx = rng.choice(len(X_train), 100, replace=False)
test_idx = rng.choice(len(X_val), 100, replace=False)

background = torch.tensor(X_train[bg_idx], dtype=torch.float32).to(device)
test_samples = torch.tensor(X_val[test_idx], dtype=torch.float32).to(device)

print("Calculating SHAP values using DeepExplainer...")
explainer = shap.DeepExplainer(model, background)
shap_vals = explainer.shap_values(test_samples)

if isinstance(shap_vals, list):
    shap_array = np.stack(shap_vals, axis=-1)  # Shape: (N_samples, N_features, N_classes)
elif isinstance(shap_vals, np.ndarray):
    shap_array = shap_vals

print(f"SHAP values shape: {shap_array.shape}")

In [ ]:
# 6. Global & Multi-Class Explanation Plots
X_test_np = X_val[test_idx]

# 1. Summary Plot
fig = plt.figure(figsize=(12, 8))
shap_list = [shap_array[:, :, c] for c in range(len(class_names))]
shap.summary_plot(shap_list, X_test_np, feature_names=feature_names, class_names=full_class_names, show=False)
plt.title("SHAP Summary Plot (Multi-Class Impact)", fontsize=14, pad=15)
plt.savefig(OUTPUT_DIR / "shap_summary.png", dpi=300, bbox_inches="tight")
plt.show()

# 2. Bar Plot (Global Importance)
fig = plt.figure(figsize=(10, 8))
mean_abs_shap_per_class = np.mean(np.abs(shap_array), axis=0)
overall_importance = np.mean(mean_abs_shap_per_class, axis=1)
top_indices = np.argsort(overall_importance)[::-1][:20]
top_indices_rev = top_indices[::-1]

plt.barh([feature_names[i] for i in top_indices_rev], overall_importance[top_indices_rev], color="#3498db")
plt.xlabel("Mean |SHAP Value| (Average Impact on Model Logits)")
plt.title("Top 20 Global Feature Importances", fontsize=14)
plt.savefig(OUTPUT_DIR / "shap_bar.png", dpi=300, bbox_inches="tight")
plt.show()

# 3. Beeswarm Plot (Safe Driving Impact)
fig = plt.figure(figsize=(10, 8))
shap.summary_plot(shap_array[:, :, 0], X_test_np, feature_names=feature_names, show=False)
plt.title("SHAP Beeswarm Plot (Class c0: Safe Driving Impact)", fontsize=14)
plt.savefig(OUTPUT_DIR / "shap_beeswarm.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# 7. Local Explanations (Waterfall & Force Plots)
with torch.no_grad():
    logits = model(test_samples)
    probs = torch.softmax(logits, dim=1).cpu().numpy()
    preds = np.argmax(probs, axis=1)

y_true_test = y_val[test_idx]
correct_idx = np.where(preds == y_true_test)[0][0]
misclass_idx = np.where(preds != y_true_test)[0]
mis_sample_idx = misclass_idx[0] if len(misclass_idx) > 0 else 1

for sample_num, s_idx in enumerate([correct_idx, mis_sample_idx], start=1):
    pred_c = preds[s_idx]
    base_val = explainer.expected_value[pred_c] if isinstance(explainer.expected_value, (list, np.ndarray)) else explainer.expected_value
    exp = shap.Explanation(
        values=shap_array[s_idx, :, pred_c],
        base_values=float(base_val),
        data=X_test_np[s_idx],
        feature_names=feature_names
    )
    fig = plt.figure(figsize=(10, 6))
    shap.plots.waterfall(exp, max_display=12, show=False)
    plt.title(f"SHAP Waterfall Sample {sample_num} (True: {class_names[y_true_test[s_idx]]}, Pred: {class_names[pred_c]})", fontsize=12)
    plt.savefig(OUTPUT_DIR / f"shap_waterfall_sample{sample_num}.png", dpi=300, bbox_inches="tight")
    plt.show()

# Force Plot HTML
s0_pred = preds[0]
base_val0 = explainer.expected_value[s0_pred] if isinstance(explainer.expected_value, (list, np.ndarray)) else explainer.expected_value
force_exp = shap.force_plot(
    base_value=float(base_val0),
    shap_values=shap_array[0, :, s0_pred],
    features=X_test_np[0],
    feature_names=feature_names,
    matplotlib=False
)
shap.save_html(str(OUTPUT_DIR / "shap_force_sample1.html"), force_exp)
print("Saved interactive Force Plot HTML.")

In [ ]:
# 8. Modality Attribution & Research Metadata Export
df_global = pd.DataFrame({
    "feature": feature_names,
    "mean_abs_shap": overall_importance,
    "modality": ["visual" if "vis" in f else "geometric" for f in feature_names]
}).sort_values(by="mean_abs_shap", ascending=False).reset_index(drop=True)
df_global.to_csv(OUTPUT_DIR / "global_feature_importance.csv", index=False)

df_classwise = pd.DataFrame(mean_abs_shap_per_class, index=feature_names, columns=[f"c{i}_importance" for i in range(len(class_names))])
df_classwise.index.name = "feature"
df_classwise.to_csv(OUTPUT_DIR / "classwise_importance.csv")

geo_shap_total = df_global[df_global["modality"]=="geometric"]["mean_abs_shap"].sum()
vis_shap_total = df_global[df_global["modality"]=="visual"]["mean_abs_shap"].sum()

metrics = {
    "total_samples_analyzed": int(len(test_idx)),
    "num_background_samples": int(len(bg_idx)),
    "feature_count": fused_features.shape[1],
    "visual_features_count": 320,
    "geometric_features_count": 12,
    "total_visual_importance_share": float(vis_shap_total / (vis_shap_total + geo_shap_total)),
    "total_geometric_importance_share": float(geo_shap_total / (vis_shap_total + geo_shap_total)),
    "top_5_global_features": df_global["feature"].head(5).tolist(),
    "top_3_geometric_features": df_global[df_global["modality"]=="geometric"]["feature"].head(3).tolist(),
}
with open(OUTPUT_DIR / "xai_metrics.json", "w") as f:
    json.dump(metrics, f, indent=4)

print("--- Milestone 6 Completed Successfully ---")
for file in sorted(os.listdir(OUTPUT_DIR)):
    print(f"Saved Artifact: {file} ({os.path.getsize(OUTPUT_DIR / file):,} bytes)")